# LLM Results Analysis

In [35]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")

In [36]:
base = Path('.')
rows = []

for path in sorted(base.glob('*_llm_results.jsonl')):
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        row['source_file'] = path.name
        rows.append(row)

df = pd.DataFrame(rows)
df = df.sort_values(['dataset', 'mode', 'k', 'num_rules']).reset_index(drop=True)

print(f'Total runs: {len(df)}')
print('Files loaded:')
for name in sorted(df['source_file'].unique()):
    print('-', name)

display(df.head())

Total runs: 410
Files loaded:
- Chinatown_baseline_llm_results.jsonl
- Chinatown_noPrototype_llm_results.jsonl
- Chinatown_rulebased_llm_results.jsonl
- ECG200_baseline_llm_results.jsonl
- ECG200_rulebased_llm_results.jsonl
- SonyAIBORobotSurface1_baseline_llm_results.jsonl
- SonyAIBORobotSurface1_rulebased_llm_results.jsonl
- UMD_baseline_llm_results.jsonl
- UMD_rulebased_llm_results.jsonl


,dataset,mode,classifier,llm,k,num_rules,accuracy,extracted_rules,instance,source_file
0,Chinatown,baseline,miniRocket,gpt-5.1,3,0,0.7,{},"[{'instance_id': 1, 'ts_idx': 14, 'true_label'...",Chinatown_baseline_llm_results.jsonl
1,Chinatown,baseline,miniRocket,gpt-5.1,3,0,0.9,{},"[{'instance_id': 1, 'ts_idx': 0, 'true_label':...",Chinatown_baseline_llm_results.jsonl
2,Chinatown,baseline,miniRocket,gpt-5.1,3,0,0.5,{},"[{'instance_id': 1, 'ts_idx': 53, 'true_label'...",Chinatown_baseline_llm_results.jsonl
3,Chinatown,baseline,miniRocket,gpt-5.1,3,0,0.9,{},"[{'instance_id': 1, 'ts_idx': 206, 'true_label...",Chinatown_baseline_llm_results.jsonl
4,Chinatown,baseline,miniRocket,gpt-5.1,3,0,0.8,{},"[{'instance_id': 1, 'ts_idx': 96, 'true_label'...",Chinatown_baseline_llm_results.jsonl


In [37]:
run_counts = (
    df.groupby(['dataset', 'mode'])
      .size()
      .reset_index(name='n_runs')
      .sort_values(['dataset', 'mode'])
)

display(run_counts)

,dataset,mode,n_runs
0,Chinatown,baseline,20
1,Chinatown,noPrototype,10
2,Chinatown,rulebased,80
3,ECG200,baseline,20
4,ECG200,rulebased,80
5,SonyAIBORobotSurface1,baseline,20
6,SonyAIBORobotSurface1,rulebased,80
7,UMD,baseline,20
8,UMD,rulebased,80


## Mean Accuracy Per Configuration

In [38]:
summary = (
    df.groupby(['dataset', 'mode', 'k', 'num_rules'])['accuracy']
      .agg(['mean', 'std', 'count', 'min', 'max'])
      .reset_index()
      .sort_values(['dataset', 'mode', 'k', 'num_rules'])
)

summary['mean'] = summary['mean'].round(3)
summary['std'] = summary['std'].round(3)
summary['min'] = summary['min'].round(3)
summary['max'] = summary['max'].round(3)

display(summary)

,dataset,mode,k,num_rules,mean,std,count,min,max
0,Chinatown,baseline,3,0,0.73,0.183,10,0.4,0.9
1,Chinatown,baseline,4,0,0.80,0.194,10,0.4,1.0
2,Chinatown,noPrototype,3,3,0.86,0.196,10,0.5,1.0
3,Chinatown,rulebased,3,2,0.68,0.235,10,0.3,0.9
4,Chinatown,rulebased,3,3,0.77,0.254,10,0.2,1.0
5,Chinatown,rulebased,3,4,0.85,0.151,10,0.6,1.0
6,Chinatown,rulebased,3,5,0.76,0.207,10,0.4,1.0
7,Chinatown,rulebased,4,2,0.84,0.184,10,0.4,1.0
8,Chinatown,rulebased,4,3,0.90,0.094,10,0.7,1.0
9,Chinatown,rulebased,4,4,0.76,0.184,10,0.5,1.0


In [39]:
for dataset in sorted(summary['dataset'].unique()):
    print(dataset)
    display(summary[summary['dataset'] == dataset].reset_index(drop=True))

Chinatown


,dataset,mode,k,num_rules,mean,std,count,min,max
0,Chinatown,baseline,3,0,0.73,0.183,10,0.4,0.9
1,Chinatown,baseline,4,0,0.80,0.194,10,0.4,1.0
2,Chinatown,noPrototype,3,3,0.86,0.196,10,0.5,1.0
3,Chinatown,rulebased,3,2,0.68,0.235,10,0.3,0.9
4,Chinatown,rulebased,3,3,0.77,0.254,10,0.2,1.0
5,Chinatown,rulebased,3,4,0.85,0.151,10,0.6,1.0
6,Chinatown,rulebased,3,5,0.76,0.207,10,0.4,1.0
7,Chinatown,rulebased,4,2,0.84,0.184,10,0.4,1.0
8,Chinatown,rulebased,4,3,0.90,0.094,10,0.7,1.0
9,Chinatown,rulebased,4,4,0.76,0.184,10,0.5,1.0


ECG200


,dataset,mode,k,num_rules,mean,std,count,min,max
0,ECG200,baseline,3,0,0.60,0.176,10,0.4,0.9
1,ECG200,baseline,4,0,0.68,0.140,10,0.5,0.9
2,ECG200,rulebased,3,2,0.76,0.135,10,0.6,1.0
3,ECG200,rulebased,3,3,0.71,0.202,10,0.3,1.0
4,ECG200,rulebased,3,4,0.56,0.184,10,0.3,0.8
5,ECG200,rulebased,3,5,0.69,0.238,10,0.1,0.9
6,ECG200,rulebased,4,2,0.43,0.142,10,0.3,0.7
7,ECG200,rulebased,4,3,0.55,0.222,10,0.2,0.8
8,ECG200,rulebased,4,4,0.58,0.257,10,0.1,1.0
9,ECG200,rulebased,4,5,0.41,0.197,10,0.2,0.8


SonyAIBORobotSurface1


,dataset,mode,k,num_rules,mean,std,count,min,max
0,SonyAIBORobotSurface1,baseline,3,0,0.61,0.173,10,0.3,0.8
1,SonyAIBORobotSurface1,baseline,4,0,0.56,0.246,10,0.1,0.9
2,SonyAIBORobotSurface1,rulebased,3,2,0.57,0.241,10,0.1,0.8
3,SonyAIBORobotSurface1,rulebased,3,3,0.62,0.215,10,0.3,1.0
4,SonyAIBORobotSurface1,rulebased,3,4,0.59,0.273,10,0.1,1.0
5,SonyAIBORobotSurface1,rulebased,3,5,0.69,0.277,10,0.2,1.0
6,SonyAIBORobotSurface1,rulebased,4,2,0.42,0.175,10,0.2,0.7
7,SonyAIBORobotSurface1,rulebased,4,3,0.48,0.204,10,0.1,0.8
8,SonyAIBORobotSurface1,rulebased,4,4,0.57,0.291,10,0.1,0.9
9,SonyAIBORobotSurface1,rulebased,4,5,0.52,0.249,10,0.1,0.8


UMD


,dataset,mode,k,num_rules,mean,std,count,min,max
0,UMD,baseline,3,0,0.85,0.165,10,0.5,1.0
1,UMD,baseline,4,0,0.90,0.156,10,0.5,1.0
2,UMD,rulebased,3,2,0.81,0.173,10,0.4,1.0
3,UMD,rulebased,3,3,0.65,0.217,10,0.3,0.9
4,UMD,rulebased,3,4,0.67,0.221,10,0.3,1.0
5,UMD,rulebased,3,5,0.69,0.213,10,0.3,1.0
6,UMD,rulebased,4,2,0.76,0.158,10,0.4,0.9
7,UMD,rulebased,4,3,0.76,0.135,10,0.6,1.0
8,UMD,rulebased,4,4,0.76,0.178,10,0.5,1.0
9,UMD,rulebased,4,5,0.72,0.199,10,0.3,0.9


In [40]:
for dataset in sorted(summary['dataset'].unique()):
    print('=' * 80)
    print(dataset)

    dataset_summary = summary[summary['dataset'] == dataset]
    best_rule = dataset_summary[dataset_summary['mode'] == 'rulebased'].sort_values('mean', ascending=False).iloc[0]
    print(f"Best rulebased config: k={int(best_rule['k'])}, r={int(best_rule['num_rules'])}, mean={best_rule['mean']:.3f}, std={best_rule['std']:.3f}")

    baseline_rows = dataset_summary[dataset_summary['mode'] == 'baseline'].sort_values('k')
    for _, row in baseline_rows.iterrows():
        print(f"Baseline k={int(row['k'])}: mean={row['mean']:.3f}")

    no_proto_rows = dataset_summary[dataset_summary['mode'] == 'noPrototype'].sort_values(['k', 'num_rules'])
    for _, row in no_proto_rows.iterrows():
        print(f"noPrototype k={int(row['k'])}, r={int(row['num_rules'])}: mean={row['mean']:.3f}")

    if not baseline_rows.empty:
        best_baseline = baseline_rows.sort_values('mean', ascending=False).iloc[0]
        print(f"Delta best_rulebased - best_baseline: {(best_rule['mean'] - best_baseline['mean']):+.3f}")

    if not no_proto_rows.empty:
        best_no_proto = no_proto_rows.sort_values('mean', ascending=False).iloc[0]
        print(f"Delta best_rulebased - best_noPrototype: {(best_rule['mean'] - best_no_proto['mean']):+.3f}")


Chinatown
Best rulebased config: k=4, r=5, mean=0.930, std=0.082
Baseline k=3: mean=0.730
Baseline k=4: mean=0.800
noPrototype k=3, r=3: mean=0.860
Delta best_rulebased - best_baseline: +0.130
Delta best_rulebased - best_noPrototype: +0.070
ECG200
Best rulebased config: k=3, r=2, mean=0.760, std=0.135
Baseline k=3: mean=0.600
Baseline k=4: mean=0.680
Delta best_rulebased - best_baseline: +0.080
SonyAIBORobotSurface1
Best rulebased config: k=3, r=5, mean=0.690, std=0.277
Baseline k=3: mean=0.610
Baseline k=4: mean=0.560
Delta best_rulebased - best_baseline: +0.080
UMD
Best rulebased config: k=3, r=2, mean=0.810, std=0.173
Baseline k=3: mean=0.850
Baseline k=4: mean=0.900
Delta best_rulebased - best_baseline: -0.090


## LaTeX Tables By k

In [41]:
dataset_order = ['Chinatown', 'UMD', 'ECG200', 'SonyAIBORobotSurface1']
dataset_labels = {
    'Chinatown': 'Chinatown',
    'UMD': 'UMD',
    'ECG200': 'ECG200',
    'SonyAIBORobotSurface1': 'Sony..1',
}

table_summary = summary.copy()
table_summary['config'] = table_summary.apply(
    lambda row: 'baseline'
    if row['mode'] == 'baseline'
    else (f'random (r={int(row["num_rules"])})' if row['mode'] == 'noPrototype' else f'r={int(row["num_rules"])}'),
    axis=1
)

all_k_values = sorted(table_summary['k'].unique())

for k_value in all_k_values:
    subset = table_summary[table_summary['k'] == k_value].copy()

    row_order = ['baseline']
    row_order += sorted(subset.loc[subset['mode'] == 'noPrototype', 'config'].unique(), key=lambda x: int(x.split('=')[-1].rstrip(')')))
    row_order += sorted(subset.loc[subset['mode'] == 'rulebased', 'config'].unique(), key=lambda x: int(x.split('=')[-1]))

    if not row_order:
        continue

    table_df = (
        subset.pivot(index='config', columns='dataset', values='mean')
        .reindex(index=row_order)
        .reindex(columns=dataset_order)
        .rename(columns=dataset_labels)
    )
    table_df.index.name = 'Config'

    print(f'Mean accuracy for k={int(k_value)}')
    display(table_df.style.format(lambda x: f'{x:.0%}' if pd.notna(x) else '-'))
    print()
    latex_table = table_df.to_latex(
        escape=False,
        na_rep='-',
        column_format='l' + 'c' * len(table_df.columns),
        float_format=lambda x: f'{x:.0%}',
        caption=f'Mean accuracy for k={int(k_value)}',
        label=f'tab:mean_accuracy_k_{int(k_value)}'
    )
    print(latex_table)
    print('\n' + '=' * 100 + '\n')

Mean accuracy for k=3


dataset,Chinatown,UMD,ECG200,Sony..1
Config,,,,
baseline,73%,85%,60%,61%
random (r=3),86%,-,-,-
r=2,68%,81%,76%,57%
r=3,77%,65%,71%,62%
r=4,85%,67%,56%,59%
r=5,76%,69%,69%,69%



\begin{table}
\caption{Mean accuracy for k=3}
\label{tab:mean_accuracy_k_3}
\begin{tabular}{lcccc}
\toprule
dataset & Chinatown & UMD & ECG200 & Sony..1 \\
Config &  &  &  &  \\
\midrule
baseline & 73% & 85% & 60% & 61% \\
random (r=3) & 86% & - & - & - \\
r=2 & 68% & 81% & 76% & 57% \\
r=3 & 77% & 65% & 71% & 62% \\
r=4 & 85% & 67% & 56% & 59% \\
r=5 & 76% & 69% & 69% & 69% \\
\bottomrule
\end{tabular}
\end{table}



Mean accuracy for k=4


dataset,Chinatown,UMD,ECG200,Sony..1
Config,,,,
baseline,80%,90%,68%,56%
r=2,84%,76%,43%,42%
r=3,90%,76%,55%,48%
r=4,76%,76%,58%,57%
r=5,93%,72%,41%,52%



\begin{table}
\caption{Mean accuracy for k=4}
\label{tab:mean_accuracy_k_4}
\begin{tabular}{lcccc}
\toprule
dataset & Chinatown & UMD & ECG200 & Sony..1 \\
Config &  &  &  &  \\
\midrule
baseline & 80% & 90% & 68% & 56% \\
r=2 & 84% & 76% & 43% & 42% \\
r=3 & 90% & 76% & 55% & 48% \\
r=4 & 76% & 76% & 58% & 57% \\
r=5 & 93% & 72% & 41% & 52% \\
\bottomrule
\end{tabular}
\end{table}



